## SOLID Design Principles

S : SRP - Single responsibility principle <br>
O : OCP - Open closed principle <br>
L : LSP - Liskov substitution principle <br>
I : ISP - Interface segregation principle <br>
D : DIP - Dependency inversion principle <br>

### **`Single responsibility principle`**

A class should have one( not exactly one, but lesser number like 2 or 3 ) reason to change. If a class mixes multiple concerns, split them.

In [21]:
# Bad Example
class Report:
    def generate_content(self) -> str:
        return "content"

    def print_to_pdf(self) -> None:
        # PDF formatting
        pass

    def save_to_file(self) -> None:
        # file I/O
        pass

In [22]:
# Good Example
class Report:
    def generate_content(self) -> str:
        return "content"


class PDFPrinter:
    def print(self, report: Report) -> None:
        # PDF formatting
        pass


class FileStorage:
    def save(self, content: str) -> None:
        # file I/O
        pass

### **`Open closed principle`**

Classes should be open for extension but closed for modification. You should be able to add new behavior without changing existing code. This usually means using interfaces or abstract classes so you can add new implementations without touching the original code

In [23]:
# Bad Example
from enum import Enum
import math

class ShapeType(Enum):
    CIRCLE = "circle"
    RECTANGLE = "rectangle"

class Shape:
    def __init__(
        self,
        shape_type: ShapeType,
        radius: float = 0,
        height: float = 0,
        width: float = 0,
    ):
        self.type = shape_type
        self.radius = radius
        self.height = height
        self.width = width

    def calculate_area(self) -> float:
        # Warning: ugly stuff below
        if self.type == ShapeType.CIRCLE:
            return math.pi * self.radius**2
        elif self.type == ShapeType.RECTANGLE:
            return self.height * self.width
        else:
            raise ValueError("Unsupported shape type.")

# Example usage
circle = Shape(ShapeType.CIRCLE, radius=5)
rectangle = Shape(ShapeType.RECTANGLE, height=4, width=6)

print(f"Circle Area: {circle.calculate_area()}")  # Circle Area: 78.53981633974483
print(f"Rectangle Area: {rectangle.calculate_area()}")  # Rectangle Area: 24

Circle Area: 78.53981633974483
Rectangle Area: 24


In [24]:
from abc import ABC, abstractmethod
import math

# Abstract Shape class to define what methods each concrete shape class should implement
class Shape(ABC):
    @abstractmethod
    def calculate_area(self) -> float:
        pass

class Circle(Shape):
    def __init__(self, radius: float):
        self.radius = radius

    def calculate_area(self) -> float:
        return math.pi * self.radius**2

class Rectangle(Shape):
    def __init__(self, width: float, height: float):
        self.width = width
        self.height = height

    def calculate_area(self) -> float:
        return self.height * self.width

# Example useage
circle = Circle(10)
rectangle = Rectangle(5, 10)

print(f"Circle area: {circle.calculate_area()}")
print(f"Rectangle area: {rectangle.calculate_area()}")


Circle area: 314.1592653589793
Rectangle area: 50


### **`Liskov Substitution Principle`**

Subclasses must work wherever the base class works. If you have a method that accepts a Bird, passing in a Penguin shouldn't break things even though penguins can't fly. 

In [25]:
# Bad Example

from abc import ABC, abstractmethod

# Abstract Shape class
class Shape(ABC):
    @abstractmethod
    def area(self) -> float:
        """Return the area of the shape."""
        pass

# Rectangle class inheriting from Shape
class Rectangle(Shape):
    def __init__(self, width: float = 0.0, height: float = 0.0):
        self._width = width
        self._height = height

    @property
    def width(self) -> float:
        return self._width

    @width.setter
    def width(self, value: float):
        self._width = value

    @property
    def height(self) -> float:
        return self._height

    @height.setter
    def height(self, value: float):
        self._height = value

    def area(self) -> float:
        return self.width * self.height

# Square class inheriting from Rectangle (LSP violation)
class Square(Rectangle):
    def __init__(self, side: float = 0.0):
        super().__init__(side, side)

    @Rectangle.width.setter
    def width(self, value: float):
        """Set both width and height to the same value."""
        self._width = value
        self._height = value

    @Rectangle.height.setter
    def height(self, value: float):
        """Set both height and width to the same value."""
        self._height = value
        self._width = value

# Example usage (LSP violation in action)
# rect = Rectangle()
# rect.width = 5
# rect.height = 10
# print("Expected area = 10 * 5 = 50.")
# print("Calculated area = " + str(rect.area()))

# Substituting Rectangle with Square (LSP violation)
rect = Square()
rect.width = 5  # Setting width will also set height to 5
rect.height = 10  # Setting height will also set width to 10
print("Expected area = 5 * 5 = 25.")  # This is misleading!
print("Calculated area = " + str(rect.area()))  # Output: 100



Expected area = 5 * 5 = 25.
Calculated area = 100


In [26]:
# Good use of LSP

from abc import ABC, abstractmethod

# Abstract Shape class
class Shape(ABC):
    @abstractmethod
    def area(self) -> float:
        """Return the area of the shape."""
        pass

# Rectangle class inheriting from Shape
class Rectangle(Shape):
    def __init__(self, width: float = 0.0, height: float = 0.0):
        self.width = width
        self.height = height

    def area(self) -> float:
        """Calculate the area of the rectangle."""
        return self.width * self.height

class Square(Shape):
    def __init__(self, side: float = 0.0):
        self.side = side

    def area(self):
        return self.side * self.side

# Example usage
rect = Rectangle()
rect.width = 5
rect.height = 10
print("Expected area = 10 * 5 = 50.")
print("Calculated area = " + str(rect.area()))

square = Square()
square.side = 5
print("Expected area = 5 * 5 = 25.")
print("Calculated area = " + str(square.area()))


Expected area = 10 * 5 = 50.
Calculated area = 50
Expected area = 5 * 5 = 25.
Calculated area = 25


### **`Interface Segregation Principle`**

Prefer small, focused interfaces over large, general-purpose ones. Don't force classes to implement methods they don't need. If a class only needs two methods from an interface with ten methods, that interface is too big.

In [27]:
# Bad example

from abc import ABC, abstractmethod
import math

class Shape(ABC):
    @abstractmethod
    def area(self):
        pass

    @abstractmethod
    def volume(self):
        pass

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return math.pi * (self.radius**2)

    def volume(self):
        raise NotImplementedError("Volume not applicable for 2D shapes.")

class Sphere(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return 4 * math.pi * (self.radius**2)

    def volume(self):
        return (4 / 3) * math.pi * (self.radius**3)

# Example usage
circle = Circle(10)
print(f"Circle area: {circle.area()}")  # Should be: 314.1592653589793

print(
    f"Circle volume: {circle.volume()}"
)  # circle has been forced to implement a method that it doesn't need

sphere = Sphere(10)
print(f"Sphere surface area: {sphere.area()}")  # Should be: 1256.6370614359173
print(f"Sphere volume: {sphere.volume()}")  # Should be: 4188.790204786391

Circle area: 314.1592653589793


NotImplementedError: Volume not applicable for 2D shapes.

In [ ]:
# Good example
from abc import ABC, abstractmethod
import math

class Shape2D(ABC):
    @abstractmethod
    def area(self):
        pass

class Shape3D(ABC):
    @abstractmethod
    def area(self):
        pass

    @abstractmethod
    def volume(self):
        pass

class Circle(Shape2D):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return math.pi * (self.radius**2)

class Sphere(Shape3D):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return 4 * math.pi * (self.radius**2)

    def volume(self):
        return (4 / 3) * math.pi * (self.radius**3)

# Example usage
circle = Circle(10)
print(f"Circle area: {circle.area()}")  # Should be: 314.1592653589793
# print(f"Circle volume: {circle.volume()}")  # If uncomment this, linter now puts a squiggly line under `volume()` method, as if to say "circle's don't have volume, what are you doing?!"

sphere = Sphere(10)
print(f"Sphere surface area: {sphere.area()}")  # Should be: 1256.6370614359173
print(f"Sphere volume: {sphere.volume()}")  # Should be: 4188.790204786391

### **`Dependency Inversion Principle`**

Dependency Inversion states that your code should depend on abstractions, not concrete implementations.

In [ ]:
# Bad Example

# DIP BAD

# Engine is our "low-level" module
class Engine:
  def start(self):
    print("Engine started")

# An alternative engine class that is FASTER than the basic engine above!
class FastEngine(Engine):
  def start(self):
    print("Activated power boot!")
    print("Fast engine started")

# Car is our "high-level" module
class Car:
  def __init__(self):
    self.engine = Engine() # car is tightly-coupled to this particular engine. If we want to give it a FastEngine, then we have to MODIFY this class, breaking the open/closed principle.

  def start(self):
    self.engine.start() 
    print("Car started")

car = Car()
car.start()

# LOGS:
# Engine started
# Car started

In [ ]:
# Good Example

from abc import ABC, abstractmethod

class Engine(ABC):
    @abstractmethod
    def start(self):
        pass

# Engine is our "low-level" module
class BasicEngine(Engine):
    def start(self):
        print("Basic engine started")

# We can now create different types of engines and inject them into car via dependency injection
class FastEngine(Engine):
    def start(self):
        print("Activated power boot!")
        print("Fast engine started")

# Car is our "high-level" module
class Car:
    def __init__(self, engine):
        self.engine = engine

    def start(self):
        self.engine.start()
        print("Car started")

fastEngine = FastEngine()  # concrete implementation to be "injected" into the car
car = Car(
    fastEngine
)  # we can now inject any class that inherits the abstract Engine class, making our code more flexible because it is no longer tightly coupled to a particular concrete engine.
car.start()

# LOGS:
# Activated power boot!
# Engine started
# Car started